In [1]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf
import sys
import importlib


In [2]:


fixed_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
#utils_path = 'C:\\Users\\Enrico Didoli\\0.Master_Thesis\\'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

general_functions_path = f'{fixed_path}General_Functions/'
if general_functions_path not in sys.path:
    sys.path.append(general_functions_path)

# Aggiungi la directory principale del progetto al percorso di sistema
csnn_path = f'{fixed_path}CSNN\\csnn\\'
if csnn_path not in sys.path:
    sys.path.append(csnn_path)


path_model_unfixed = f'{csnn_path}0.Unfixed_files'
if path_model_unfixed  not in sys.path:
    sys.path.append(path_model_unfixed )
# Ricaricali per forzare la lettura aggiornata del file


In [3]:
decache_files = ['timepoints_elaboration', 'results_elaboration', 'new_datasets_generation']


# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)


from utils import trainer, DensityDifference, dataset

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv


from timepoints_elaboration import donation_extraction, dataset_elaboration, donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from new_datasets_generation import splitting_and_dataset_elaboration, remove_labels


timepoints_elaboration rimosso dalla cache
results_elaboration non trovato nella cache
new_datasets_generation non trovato nella cache


In [4]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{fixed_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    data = pd.read_csv(file_path, sep = ';', decimal = ',').astype('float32')
    ALL_DATASETS.append(data) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for data in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [data]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [5]:
for i, data in enumerate(ALL_DATASETS):
    blast_n = (data['IsBlast'] == 1).sum()
    print(f'data {i} has {blast_n} cells over {len(data)} cells')

data 0 has 0 cells over 6558750 cells
data 1 has 0 cells over 2764877 cells
data 2 has 0 cells over 1291729 cells
data 3 has 1348 cells over 843500 cells
data 4 has 170 cells over 1404000 cells
data 5 has 0 cells over 3265250 cells
data 6 has 639 cells over 430869 cells
data 7 has 15 cells over 722000 cells
data 8 has 757 cells over 864000 cells
data 9 has 0 cells over 1947518 cells
data 10 has 308059 cells over 778750 cells
data 11 has 830101 cells over 5510750 cells
data 12 has 72 cells over 208000 cells
data 13 has 0 cells over 2912500 cells
data 14 has 3096 cells over 160500 cells
data 15 has 1449 cells over 3591480 cells
data 16 has 0 cells over 3107000 cells
data 17 has 9147 cells over 637409 cells
data 18 has 227 cells over 2928000 cells
data 19 has 518 cells over 164553 cells
data 20 has 398 cells over 191975 cells
data 21 has 0 cells over 169000 cells
data 22 has 2390 cells over 479000 cells
data 23 has 77 cells over 455000 cells
data 24 has 44697 cells over 160828 cells
data 

In [6]:
# Show patients donations
print(multiple_donations)
#['12', '9', '13', '7', '11'] 

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [7]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [9]:

# Samples donors for Train, Validation and Test sets    
train_donors_idx, val_donors_idx, test_donors_idx = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors,
                        mixed_donors, seed = 42)



Precess starts. Dividing donors...
healthy_donors_idx, blast_donors_idx, mixed_donors_idx: [0, 4, 2, 1, 3], [0, 2, 1],[4, 0, 2, 1, 3]
Seting Train, Validation and Test idx...
['12', '9', '13', '7', '11'] ['6', '15', '3'] ['2', '8', '14', '1', '4']


In [10]:

#  Retrieves specific donor datasets from all datasets list
train_datasets_extracted = donation_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
val_datasets_extracted = donation_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
test_datasets_extracted = donation_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

#print(len(train_datasets_extracted)) # list of donators
#print(len(train_datasets_extracted[0])) # list of donations
#print(len(train_datasets_extracted[0][0])) # dataset of cells

In [11]:
n_sub = 3
seed = 42
n_cells = 100000
new_train_datasets, new_train_y, new_val_datasets, new_val_y, new_test_datasets, new_test_y = splitting_and_dataset_elaboration(train_datasets_extracted, val_datasets_extracted, test_datasets_extracted, n_sub, n_cells, seed)

new_no_label_test_dataset = remove_labels(new_test_datasets)

New training datasets creation...
5
2

New Donor
2
Tot blast data in the donor timepoints: 639
Tot blast data in the donor timepoints: 654
Extraction Done
Condition: 1

New Donor
2
Tot blast data in the donor timepoints: 44697
Tot blast data in the donor timepoints: 46110
Extraction Done
Condition: 1

New Donor
1
Timepoint condition: Healthy
Extraction Done
Condition: 0

New Donor
3
Tot blast data in the donor timepoints: 518
Tot blast data in the donor timepoints: 916
Tot blast data in the donor timepoints: 916
Extraction Done
Condition: 1

New Donor
3
Tot blast data in the donor timepoints: 1348
Tot blast data in the donor timepoints: 1518
Tot blast data in the donor timepoints: 1518
Extraction Done
Condition: 1
[1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0]
Done!

New training datasets creation...

New Donor
Tot blast data in the donor timepoints: 9147
Tot blast data in the donor timepoints: 9374
Extraction Done
Condition: 1

New Donor
Timepoint co

In [ ]:
for i, dataset in enumerate(new_test_datasets):
    blast_n = (dataset['IsBlast'] == 1).sum()
    
    print(f'dataset {i} has {blast_n} cells over {len(dataset)} cells. Label: {new_test_y[i]}')

In [12]:
decache_files = ['tuning_ball_logistic.yaml', 'timepoints_elaboration', 'results_elaboration', 'new_datasets_generation']


# Rimuovi il modulo specifico dalla cache
from timepoints_elaboration import remove_from_cache
remove_from_cache(decache_files)


from utils import trainer, DensityDifference, dataset

importlib.reload(trainer)
importlib.reload(DensityDifference)
importlib.reload(dataset)

from utils import trainer, DensityDifference, dataset
#import train_logistic_cv


from timepoints_elaboration import donation_extraction, dataset_elaboration, donor_division, patient_code_extraction, remove_from_cache

from results_elaboration import extract_hyper, phenotype_prediction, default_serializer, show_hyperparameters
from results_elaboration import elaborate_predictions, show_hyper

from new_datasets_generation import splitting_and_dataset_elaboration, remove_labels


tuning_ball_logistic.yaml non trovato nella cache
timepoints_elaboration rimosso dalla cache
results_elaboration rimosso dalla cache
new_datasets_generation rimosso dalla cache


In [13]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import os


RESULT_DIR = f'{path_model_unfixed}/data/B-ALL'

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(f'{RESULT_DIR}/train', exist_ok=True)
os.makedirs(f'{RESULT_DIR}/test', exist_ok=True)

#train_set = pd.read_csv("metadata/B-ALL_train.csv")
#test_set = pd.read_csv("metadata/B-ALL_test.csv")

Index = 0
for i, data in enumerate(new_train_datasets):
    Index += 1
    feature_names = data.columns

    b_data = len(data[data['IsBlast'] == 1])
    proportion = b_data / len(data)
    data = data.drop(columns = ['IsBlast'])
    
    X = torch.tensor(data.to_numpy())
    #X = torch.tensor(data)
    y = new_train_y[i]
    data = {
        "X": X,
        "y": y,
        "idx": Index,
        "proportion": proportion,
        "tags": ["B-ALL", "train"],
        "feature_names": feature_names
    }
    path = f'{RESULT_DIR}/train/'+ f"{Index}.pt"

    torch.save(data, path)

for i, data in enumerate(new_test_datasets):
    Index += 1
    #print(i)
    feature_names = data.columns

    b_data = len(data[data['IsBlast'] == 1])
    proportion = b_data / len(data)
    data = data.drop(columns = ['IsBlast'])
    
    X = torch.tensor(data.to_numpy())
    #X = torch.tensor(data)
    y = new_train_y[i]
    
    data = {
        "X": X,
        "y": y,
        "idx": Index,
        "proportion": proportion,
        "tags": ["B-ALL", "test"],
        "feature_names": feature_names
    }
    path = f'{RESULT_DIR}/test/'+ f"{Index}.pt"

    torch.save(data, path)

In [14]:
import sys
from yaml import load, FullLoader
sys.argv = ['tuning_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\tuning_ball_logistic.yaml']

In [15]:
%%time
import numpy as np
import random
import torch
import csv
import os
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import ParameterGrid
from yaml import load, FullLoader
import sys
from tqdm import tqdm

from utils.dataset import PointsetDataset
from utils.trainer import dd_initialize, train_initialize, train_head, finetune, load_evaluate

with open(sys.argv[1], 'r') as f:
    config = load(f, Loader=FullLoader)
    #print(config)

import os
print(f'{RESULT_DIR}/train')
#print('ciao')
print(os.path.exists(f'{RESULT_DIR}/train'))
#print('ff')
#print(config['datasets'])

start_seed = config['start_seed']
end_seed = config['start_seed'] + config['n_seeds']

metrics = [
    ("accuracy", accuracy_score),
    ("f1", f1_score),
    ("precision", precision_score),
    ("recall", recall_score),
    ("auc", roc_auc_score),
]

#print(config['param_grid'])
param_grid = ParameterGrid(config['param_grid']) # makes grid search over all combinations of parameters (originally 80)
sample = param_grid[0]

keys = list(sample.keys())


os.makedirs("experiment/%s/" % config['experiment_name'], exist_ok=True)
with open('experiment/%s/results.csv' % config['experiment_name'], 'w') as f:
    writer = csv.writer(f)
    row_names = ["seed"]
    writer.writerow(row_names + list(sample.keys()) + ["param_hash"] +
                    [name + "_train" for name, _ in metrics] + 
                    [name + "_valid" for name, _ in metrics])

#param_grid = ParameterGrid(config['param_grid'])
#print(f'tm: {tqdm(param_grid)}')
#print(f'len_tm: {len(tqdm(param_grid))}')

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'start_training...')
for seed in range(1): #range(start_seed, end_seed):
    
    # retrieve train and validation data (dataset.py)
    dataset_train = PointsetDataset(config['datasets'], train=True, fold=1, random_state=seed, **config['dataset_config'])
    dataset_valid = PointsetDataset(config['datasets'], train=False, fold=1, random_state=seed, **config['dataset_config'])

    #print(dataset_valid)

    # trains the initial scores using hernel density
    diff, allpos_sample, allneg_sample = dd_initialize(dataset_train, dataset_valid, **config['experiment_config'])
    print(f'Number of iterations in the folder: {len(tqdm(param_grid))}')
    for params in tqdm(param_grid): # for every combination of parameters 
        
        hashparams = params.copy()
        hashparams['architecture'] = frozenset(hashparams['architecture'])
        param_hash = hash(frozenset(hashparams.items()))
        print(param_hash)

        resdir = "experiment/%s/models/seed_%d/hash_%d" % (config['experiment_name'], seed, param_hash) #results directory

        os.makedirs(resdir, exist_ok=True)

        if True:
            kwargs = params.copy()
            for key in config['experiment_config']:
                kwargs[key] = config['experiment_config'][key]

            # train the model
            model, Xinit_train, yinit_train = train_initialize(dataset_train, dataset_valid, diff, allpos_sample, allneg_sample, max_iter = 1)#, **kwargs)

            torch.save((Xinit_train, yinit_train), "%s/dd_init.pt" % resdir)
            torch.save(model.cpu(), "%s/model_init.pt" % resdir)

            # train again the model with the best  parameters and weights? 
            model = train_head(dataset_train, dataset_valid, model, **config['experiment_config'])

            torch.save(model.cpu(), "%s/model_head.pt" % resdir)

            
            model, (preds_val, y_valid), (preds_train, y_train) = finetune(dataset_train, dataset_valid, model, **kwargs)

            torch.save(model, "%s/model.pt" % resdir)
        else:
            # validation
            model, (preds_val, y_valid), (preds_train, y_train) = load_evaluate(dataset_train, dataset_valid, "%s/model.pt")
            
        train_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                train_metrics.append(metric(y_train.numpy(), (preds_train > 0.5).to(int).numpy()))
            else:
                train_metrics.append(metric(y_train.numpy(), preds_train.numpy()))

        val_metrics = []
        for name, metric in metrics:
            if name in ["accuracy", "f1", "precision", "recall"]:
                val_metrics.append(metric(y_valid.numpy(), (preds_val > 0.5).to(int).numpy()))
            else:
                val_metrics.append(metric(y_valid.numpy(), preds_val.numpy()))

        #params['hash'] = param_hash
        with open('experiment/%s/results.csv' % config['experiment_name'], 'a') as f:
            writer = csv.writer(f)
            metadata = [seed]
            param_list = [str(v) for v in params.values()]
            param_list.append(param_hash)
            writer.writerow(metadata + param_list + train_metrics + val_metrics)


C:\Users\admin\0.Master_Thesis\CSNN\csnn\0.Unfixed_files/data/B-ALL/train
True
start_training...
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
ok

ok


  0%|                                                                                            | 0/9 [00:00<?, ?it/s]

1
Iteration 0: tensor([[1.6309, 1.5472, 0.7484,  ..., 2.7918, 1.0631, 2.5651],
        [0.6724, 0.7540, 0.4832,  ..., 0.7448, 0.8574, 2.5452],
        [0.8888, 0.9579, 0.4387,  ..., 0.9245, 3.1891, 2.6491],
        ...,
        [1.1768, 1.1958, 0.6174,  ..., 2.4728, 1.7430, 2.8646],
        [1.1240, 1.1101, 0.5444,  ..., 2.1557, 3.4699, 1.3201],
        [1.0382, 1.1676, 0.6126,  ..., 0.4659, 0.8168, 1.1784]])

ok
1
Iteration 1: tensor([[0.6309, 0.7254, 0.4216,  ..., 0.5429, 0.7654, 0.9335],
        [0.6315, 0.6967, 0.4125,  ..., 0.8065, 0.6105, 0.6832],
        [2.1416, 1.4699, 1.5113,  ..., 2.8962, 1.1728, 2.0508],
        ...,
        [1.8285, 1.6182, 2.4659,  ..., 1.6162, 0.7794, 2.1054],
        [0.5408, 0.5549, 0.4510,  ..., 1.3581, 0.8779, 1.8501],
        [0.9786, 1.0100, 0.6197,  ..., 1.1752, 0.7742, 2.6965]])

ok
1
Iteration 2: tensor([[1.6890, 1.4717, 1.5078,  ..., 1.2730, 1.1575, 2.6417],
        [1.8048, 1.6216, 1.4978,  ..., 1.3536, 0.8761, 2.4076],
        [0.7574, 0.7610

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 12.50it/s]


1
Iteration 8: tensor([[2.1797, 2.4127, 2.6449,  ..., 1.5142, 1.0952, 2.4297],
        [2.3846, 2.5191, 2.2271,  ..., 1.3195, 0.8414, 2.1277],
        [0.8022, 0.8742, 0.5185,  ..., 0.5851, 0.5751, 1.3243],
        ...,
        [0.8578, 0.8687, 0.5645,  ..., 0.9131, 0.4299, 2.6848],
        [2.0261, 2.1135, 3.4799,  ..., 1.3167, 1.1543, 2.2675],
        [1.5463, 1.6354, 0.8556,  ..., 2.7073, 0.5716, 1.9261]])

ok


...

-2918908910604671198
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.6928, tr_acc: 0.5176
[Epoch 0] tr_loss: 0.7391, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.6994, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.7249, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.6866, tr_acc: 0.5000
Best accuracy: 0.52
[Epoch 0] tr_loss: 0.6872, tr_acc: 0.5714, te_loss: 0.7190, te_acc: 0.5000
[Epoch 250] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 500] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 750] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 1000] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 1001] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 0] tr_loss: 0.6836, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000
[Epoch 41] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6989, te_acc: 0.5000


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
 88%|█████████████████████████████████████████████████████████████████████████▌          | 7/8 [01:35<00:16, 16.58s/it]

-6773661278049627198
Initializing models with 5 restarts
[Epoch 0] tr_loss: 0.6944, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.7030, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.7353, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.6961, tr_acc: 0.5000
[Epoch 0] tr_loss: 0.7180, tr_acc: 0.5000
Best accuracy: 0.50
[Epoch 0] tr_loss: 0.7067, tr_acc: 0.5714, te_loss: 0.7566, te_acc: 0.5000
[Epoch 250] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 500] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 750] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 1000] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 1001] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 0] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000
[Epoch 13] tr_loss: 0.6835, tr_acc: 0.5714, te_loss: 0.6990, te_acc: 0.5000


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [01:43<00:00, 12.95s/it]

CPU times: total: 2min 33s
Wall time: 7min 37s


In [16]:
import pandas as pd

# Leggi il CSV
df = pd.read_csv(f'{fixed_path}CSNN/experiment/tuning_ball_logistic/results.csv')

# Visualizza tutto
print(df)


   seed negative_penalty    lr  dd_threshold  architecture  \
0     0           [3, 1]  95.0        0.0010             1   
1     0           [3, 1]  95.0        0.0001             1   
2     0           [3, 1]  97.5        0.0010             1   
3     0           [3, 1]  97.5        0.0001             1   
4     0           [6, 1]  95.0        0.0010             1   
5     0           [6, 1]  95.0        0.0001             1   
6     0           [6, 1]  97.5        0.0010             1   
7     0           [6, 1]  97.5        0.0001             1   

            param_hash  accuracy_train  f1_train  precision_train  \
0 -4781901312666028972        0.571429       0.0              0.0   
1 -1202501418423672588        0.571429       0.0              0.0   
2   169734510484782055        0.571429       0.0              0.0   
3  5885681106989479367        0.571429       0.0              0.0   
4  8631623603102478005        0.571429       0.0              0.0   
5  6846416831723006101     

In [17]:

# Visualizza le prime righe
print(df.head())



   seed negative_penalty    lr  dd_threshold  architecture  \
0     0           [3, 1]  95.0        0.0010             1   
1     0           [3, 1]  95.0        0.0001             1   
2     0           [3, 1]  97.5        0.0010             1   
3     0           [3, 1]  97.5        0.0001             1   
4     0           [6, 1]  95.0        0.0010             1   

            param_hash  accuracy_train  f1_train  precision_train  \
0 -4781901312666028972        0.571429       0.0              0.0   
1 -1202501418423672588        0.571429       0.0              0.0   
2   169734510484782055        0.571429       0.0              0.0   
3  5885681106989479367        0.571429       0.0              0.0   
4  8631623603102478005        0.571429       0.0              0.0   

   recall_train  auc_train  accuracy_valid  f1_valid  precision_valid  \
0           0.0   0.476852             0.5       0.0              0.0   
1           0.0   0.509259             0.5       0.0              

In [18]:
# Visualizza statistiche
print(df.describe())



       seed         lr  dd_threshold  architecture    param_hash  \
count   8.0   8.000000      8.000000           8.0  8.000000e+00   
mean    0.0  96.250000      0.000550           1.0  7.320604e+17   
std     0.0   1.336306      0.000481           0.0  5.738046e+18   
min     0.0  95.000000      0.000100           1.0 -6.773661e+18   
25%     0.0  95.000000      0.000100           1.0 -3.384657e+18   
50%     0.0  96.250000      0.000550           1.0 -5.163835e+17   
75%     0.0  97.500000      0.001000           1.0  6.125865e+18   
max     0.0  97.500000      0.001000           1.0  8.631624e+18   

       accuracy_train  f1_train  precision_train  recall_train  auc_train  \
count        8.000000       8.0              8.0           8.0   8.000000   
mean         0.571429       0.0              0.0           0.0   0.379051   
std          0.000000       0.0              0.0           0.0   0.167572   
min          0.571429       0.0              0.0           0.0   0.175926   
25

In [19]:
# Trova il modello migliore per AUC validation
best_model = df.loc[df['auc_valid'].idxmax()]
print("\nBest model:")
print(best_model)

# recostruct path for the best model
base_path = f'{fixed_path}CSNN/experiment/tuning_ball_logistic/models/'
model_seed = best_model['seed']
model_hash = best_model['param_hash']
best_model_path_recostructed = f'{base_path}/seed_{model_seed}/hash_{model_hash}/'
print(best_model_path_recostructed)



Best model:
seed                                   0
negative_penalty                  [6, 1]
lr                                  97.5
dd_threshold                       0.001
architecture                           1
param_hash          -2918908910604671198
accuracy_train                  0.571429
f1_train                             0.0
precision_train                      0.0
recall_train                         0.0
auc_train                       0.305556
accuracy_valid                       0.5
f1_valid                             0.0
precision_valid                      0.0
recall_valid                         0.0
auc_valid                       0.888889
Name: 6, dtype: object
C:\Users\admin\0.Master_Thesis\CSNN/experiment/tuning_ball_logistic/models//seed_0/hash_-2918908910604671198/


In [20]:
# Ordina per una metrica specifica
df_sorted = df.sort_values('auc_valid', ascending=False)
print("\nTop 5 models:")
print(df_sorted.head())


Top 5 models:
   seed negative_penalty    lr  dd_threshold  architecture  \
6     0           [6, 1]  97.5        0.0010             1   
3     0           [3, 1]  97.5        0.0001             1   
4     0           [6, 1]  95.0        0.0010             1   
0     0           [3, 1]  95.0        0.0010             1   
2     0           [3, 1]  97.5        0.0010             1   

            param_hash  accuracy_train  f1_train  precision_train  \
6 -2918908910604671198        0.571429       0.0              0.0   
3  5885681106989479367        0.571429       0.0              0.0   
4  8631623603102478005        0.571429       0.0              0.0   
0 -4781901312666028972        0.571429       0.0              0.0   
2   169734510484782055        0.571429       0.0              0.0   

   recall_train  auc_train  accuracy_valid  f1_valid  precision_valid  \
6           0.0   0.305556             0.5       0.0              0.0   
3           0.0   0.263889             0.5       0.

In [22]:
#dataset_test = PointsetDataset(config['datasets'], train=False, fold=0, random_state=seed) #, **config['dataset_config'])
#print(os.path.exists(f'{path_unfixed}/data/B-ALL/test'))

sys.argv = ['test_ball_logistic.yaml', r'C:\Users\admin\0.Master_Thesis\CSNN\csnn\config\test_ball_logistic.yaml']
with open(sys.argv[1], 'r') as f:
    test_config = load(f, Loader=FullLoader)
    print(test_config)
#test_path = f'{path_unfixed}/data/B-ALL/test'
print(test_config['datasets'])


#FIXARE TESTING
dataset_test = PointsetDataset(test_config['datasets'], train=False, fold=None, random_state=seed, **test_config['dataset_config'])
#print(dataset_test.X)
#print(dataset_test.y)


{'experiment_name': 'test_ball_logistic', 'start_seed': 0, 'n_seeds': 1, 'datasets': ['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/'], 'dataset_config': {'sample_size': 100000}}
['C:\\Users\\admin\\0.Master_Thesis\\CSNN\\csnn/0.Unfixed_files/data/B-ALL/test/']


In [23]:
import hashlib
def test_load_evaluate(dataset_test, fn):
    x_test = dataset_test.X
    y_test = dataset_test.y
    
    x_test = x_test.to(device).float()
    y_test = y_test.to(device).float()

    model = torch.load(fn)
    model = model.to(device)
    model.eval()

    with torch.set_grad_enabled(False):
        preds_test = model(x_test)
        preds_test = preds_test.squeeze(1)

    return model.cpu(), (preds_test.cpu(), y_test.cpu())
    

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


model, (preds_test, y_test) = test_load_evaluate(dataset_test,  f'{best_model_path_recostructed}model.pt')
print(f'Test Predictions: {preds_test}')
print(f'True Labels: {y_test}')
        



Test Predictions: tensor([0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461,
        0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461,
        0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461,
        0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461, 0.4461])
True Labels: tensor([1., 1., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 0., 0., 1., 1., 1.,
        0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 1., 1., 1., 0., 0., 0.])
